### 4.0 Document Loaders

There are many other types of Documents that can be loaded in. You can see all the document loaders available here: https://python.langchain.com/docs/modules/data_connection/document_loaders/

Keep in mind many Loaders are dependent on other libraries, meaning issues in those libraries can end up breaking the Langchain loaders.

#### 4.1 CSV Loader

In [ ]:
from langchain_community.document_loaders import CSVLoader

In [ ]:
loader = CSVLoader(r'penguins.csv')
data = loader.load()

In [ ]:
print(data[1].page_content)

#### 4.2 HTML 

In [ ]:
!pip install -U lxml

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader

In [ ]:
loader = BSHTMLLoader(r'some_website.html')
data = loader.load()
data

#### 4.3 PDF

In [ ]:
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader(r'report.pdf')
pages = loader.load_and_split()

In [ ]:
print(pages[0].page_content)

#### 4.4 Document Tranformations: Split by Character, split by tokens

In [ ]:
with open(r'FDR_State_of_Union_1944.txt') as file:
    speech_text = file.read()

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
text_splitter = CharacterTextSplitter(separator="\n\n",chunk_size=300) #1000 is default value
text_splitter

In [ ]:
texts = text_splitter.create_documents([speech_text])
print(type(texts))
print('\n')
print(texts[:10])

In [ ]:
type(texts[0]), texts[0]

In [ ]:
!pip install tiktoken

In [ ]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size = 200) #now chunk size is a hard length based on tokens

In [ ]:
texts = text_splitter.split_text(speech_text)

In [ ]:
texts[0:4], len(texts)

#### 4.5 Text Embeddings 

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [ ]:
from langchain_openai import OpenAI

f = open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-8\key-vault\openai\api.key")
apikey = f.read()
f.close()

In [ ]:
embeddings = OpenAIEmbeddings(api_key=apikey)

In [ ]:
text = "Some normal text to send to OpenAI to be embedded into a N dimensional vector"

In [ ]:
embedded_text = embeddings.embed_query(text)

In [ ]:
embedded_text[0:9]

### 5.0 Vector Store

##### We can save the embeddings into a Vector store - Chroma

In [ ]:
#!pip install langchain_chroma

In [ ]:
import chromadb
print(chromadb.__version__)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader

##### Load the document and split(still recommended even if under the context window)

In [ ]:
# load the document and split it into chunks
loader = TextLoader(r'FDR_State_of_Union_1944.txt')
documents = loader.load()

# f = open(r"FDR.txt")
# data = f.read()
# f.close()

documents

In [ ]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=200)
docs = text_splitter.split_documents(documents)

##### Connect to OpenAI for Embeddings

In [ ]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

##### Pass Embeddings and Docs into Chroma

In [ ]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function, persist_directory=r'speech_embedding_db')

##### Save the new embeddings to the disk

In [ ]:
# Helpful to force a save
db.persist()

##### Loading embeddings from the disk

In [ ]:
persist_directory=r'speech_embedding_db'
db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)

In [ ]:
new_doc = "What did FDR say about the cost of food law?"
docs = db_connection.similarity_search(new_doc)

In [ ]:
print(docs[0].page_content)

##### Adding a new document

In [ ]:
# load the document and split it into chunks
loader = TextLoader(r"Lincoln_State_of_Union_1862.txt")
documents = loader.load()

In [ ]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=200)
docs = text_splitter.split_documents(documents)

In [ ]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)

In [ ]:
docs = db.similarity_search('slavery')

In [ ]:
print(docs[0].page_content)

### 5.1 Vector Store Retriever

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.vectorstores import Chroma

In [ ]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

In [ ]:
# db_connection = Chroma(persist_directory=r'mk_ultra',embedding_function=embedding_function)
persist_directory=r'speech_embedding_db'
db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)

In [ ]:
retriever = db_connection.as_retriever()

In [ ]:
search_kwargs = {"score_threshold":0.5,"k":5}
docs = retriever.invoke("slavery",search_kwargs)

In [ ]:
docs 

### 6.0 Exercise : Vector Stores

###  Data Connections Exercise

#### Ask a Legal Research Assistant Bot about the US Constitution

Let's revisit our first exercise and add offline capability using ChromaDB. Your function should do the following:

* Read the US_Constitution.txt file inside the some_data folder
* Split this into chunks (you choose the size)
* Write this to a ChromaDB Vector Store
* Use Context Compression to return the relevant portion of the document to the question

In [ ]:
# Build a sample vectorDB

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma



In [ ]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
import chromadb

In [ ]:
def db_setup():
    persist_directory=r"constitution"
    
    # PART ONE:
    # LOAD "/US_Constitution in a Document object
    loader = TextLoader(r"US_Constitution.txt")
    documents = loader.load()
    
    # PART TWO
    # Split the document into chunks (you choose how and what size)
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
    docs = text_splitter.split_documents(documents)
    
    # PART THREE
    # EMBED THE Documents (now in chunks) to a persisted ChromaDB
    embedding_function = OpenAIEmbeddings(api_key=apikey)
    db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)
    db.persist()
    return db

db = db_setup()

In [ ]:
llm = ChatOpenAI(temperature=0.5, api_key=apikey)

In [ ]:
def us_constitution_helper(llm, db, question):
    '''
    Takes in a question about the US Constitution and returns the most relevant
    part of the constitution. Notice it may not directly answer the actual question!
    
    Follow the steps below to fill out this function:
    '''

    # PART FOUR
    # Use ChatOpenAI and ContextualCompressionRetriever to return the most
    # relevant part of the documents.

    # results = db.similarity_search("What is the 13th Amendment?")
    # print(results[0].page_content) # NEED TO COMPRESS THESE RESULTS!
    
    compressor = LLMChainExtractor.from_llm(llm)

    compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, 
                                                           base_retriever=db.as_retriever(search_type='mmr'))

    compressed_docs = compression_retriever.invoke(question)

    return compressed_docs[0].page_content

In [ ]:
print(us_constitution_helper(llm, db, "What are the reponsibilities of congress?"))